# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cooper30/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data contract

1. **What one row means:**  
   One row in my modeling frame represents one content item for one client during the selected observation month. The raw warehouse table is daily, so I will aggregate it to one row per client × content item for March 2026.

2. **Tables used:**  
   I will primarily use `fact_content_daily_performance`.

3. **Time window:**  
   I will use March 2026 (`2026-03-01` to `2026-03-31`) as the development month. I will not use the June 2026 `_sample` table for label development because it is the final panel month.

4. **Prediction / ranking target:**  
   My lane is Refresh / Content Opportunity Scoring. The goal is to rank content items by how urgently they should be reviewed for a possible content refresh.

5. **Deliberately excluded:**  
   I will exclude label-derived and future-performance fields from the feature set because they would leak information that would not be available at the real decision moment.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/**/*.parquet'"
    f")"
)

print("Warehouse connection ready.")

Warehouse connection ready.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field roles

**Features**
- Search impressions
- Search clicks
- Average search position
- GA4 sessions or users when available
- Content age / recency-related information if available in the warehouse

**Label / proxy**
- A refresh-opportunity or declining-performance signal used to support the ranking target.

**Context**
- `client_id`
- `content_id`
- `report_date`

These fields identify the observation but are not used directly as predictive features.

**Excluded**
- Future-period performance
- Label-derived columns
- Any field directly used to construct the outcome

These are excluded to prevent target leakage.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 3 — Availability check using IS TRUE

q3 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM {FACT_DAILY}
WHERE report_date >= DATE '2026-03-01'
  AND report_date <  DATE '2026-04-01'
"""

con.execute(q3).df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows
0,9841378,413966


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [16]:
# Query 1 — Verify the raw grain for March 2026

q1 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS distinct_grain_rows
FROM {FACT_DAILY}
WHERE report_date >= DATE '2026-03-01'
  AND report_date <  DATE '2026-04-01'
"""

con.execute(q1).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows
0,9841378,9841378


In [17]:
# Query 2 — Row count and date span for March 2026

q2 = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {FACT_DAILY}
WHERE report_date >= DATE '2026-03-01'
  AND report_date <  DATE '2026-04-01'
"""

con.execute(q2).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [18]:
# Query 3 — Availability check using IS TRUE

q3 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows
FROM {FACT_DAILY}
WHERE report_date >= DATE '2026-03-01'
  AND report_date <  DATE '2026-04-01'
"""

con.execute(q3).df()

,total_rows,ga4_available_rows
0,9841378,413966


### Verification summary

- **Grain:** The March 2026 raw table is unique at `client_hash_id × content_hash_id × report_date` grain. Total rows and distinct grain rows are both 9,841,378.
- **Slice:** The March 2026 development slice contains 9,841,378 raw rows and spans from `2026-03-01` to `2026-03-31`.
- **Availability:** 413,966 of 9,841,378 rows survive `ga4_data_available IS TRUE`, which is about 4.21%.

In [19]:
# Five-feature frame — March 2026
# Context columns are kept for identification; the model features are exactly five.

feature_frame = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions_31d,
    SUM(gsc_clicks) AS clicks_31d,

    SUM(gsc_clicks) * 1.0
        / NULLIF(SUM(gsc_impressions), 0) AS ctr_31d,

    SUM(gsc_sum_position) * 1.0
        / NULLIF(SUM(gsc_impressions), 0) AS avg_position_31d,

    COUNT(DISTINCT report_date) AS observed_days_31d

FROM {FACT_DAILY}
WHERE report_date >= DATE '2026-03-01'
  AND report_date <  DATE '2026-04-01'
  AND gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id

""").df()

feature_frame

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_31d,clicks_31d,ctr_31d,avg_position_31d,observed_days_31d
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,0.000000,4.311688,24
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,0.002028,8.049866,31
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,0.000000,5.885246,27
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,0.001418,5.863830,31
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,0.000000,14.360000,21
...,...,...,...,...,...,...,...
176733,client_2b4306c3ed003f01,content_4a46b45b40a57f11,1.0,0.0,0.000000,47.000000,1
176734,client_2b4306c3ed003f01,content_9eeeb3f77716113c,1.0,0.0,0.000000,9.000000,1
176735,client_20259bd6705d81d4,content_2d8e52b436a736c8,3.0,0.0,0.000000,7.333333,1
176736,client_20259bd6705d81d4,content_4d0dafdb2450a480,15.0,0.0,0.000000,42.466667,1


### Five features and availability at decision time

1. **impressions_31d**  
   Knowable at the decision moment because it uses only GSC impressions already observed during the completed March 2026 window.

2. **clicks_31d**  
   Knowable at the decision moment because it uses only GSC clicks already recorded during the completed observation month.

3. **ctr_31d**  
   Knowable at the decision moment because it is calculated only from historical clicks and impressions available by the end of the observation window.

4. **avg_position_31d**  
   Knowable at the decision moment because it is calculated from historical Search Console position observations collected during the completed month.

5. **observed_days_31d**  
   Knowable at the decision moment because it counts only dates already present in the warehouse during the observation window.

In [20]:
# Deliberate leakage experiment
# We intentionally create a label from one of the observed fields,
# then leak that same information back into the model.

# Deliberate leakage experiment

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Copy the feature frame
leak_df = feature_frame.copy()

# --------------------------------------------------
# 1. Create a simple proxy target
# --------------------------------------------------

threshold = leak_df["impressions_31d"].median()

leak_df["refresh_proxy"] = (
    leak_df["impressions_31d"] > threshold
).astype(int)

# --------------------------------------------------
# 2. Honest model
# --------------------------------------------------

honest_features = [
    "clicks_31d",
    "ctr_31d",
    "avg_position_31d",
    "observed_days_31d"
]

X_honest = leak_df[honest_features]
y = leak_df["refresh_proxy"]

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

honest_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)

honest_score = accuracy_score(
    y_test,
    honest_predictions
)

# --------------------------------------------------
# 3. Deliberately create a label-derived leak
# --------------------------------------------------

# This column directly copies the target.
# It is intentionally invalid as a real model feature.
leak_df["refresh_proxy_leak"] = leak_df["refresh_proxy"]

leaky_features = honest_features + ["refresh_proxy_leak"]

X_leaky = leak_df[leaky_features]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

leaky_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

leaky_model.fit(X_train_l, y_train_l)

leaky_predictions = leaky_model.predict(X_test_l)

leaky_score = accuracy_score(
    y_test_l,
    leaky_predictions
)

# --------------------------------------------------
# 4. Remove the leaked column
# --------------------------------------------------

leak_df = leak_df.drop(columns=["refresh_proxy_leak"])

print(f"Rows in feature frame: {len(feature_frame):,}")
print(f"Honest score: {honest_score:.3f}")
print(f"Leaky score:  {leaky_score:.3f}")
print("Removed leaked column: refresh_proxy_leak")

Rows in feature frame: 176,738
Honest score: 0.912
Leaky score:  1.000
Removed leaked column: refresh_proxy_leak


### Deliberate leakage trap

The honest model achieved an accuracy of 0.912 using features available at the decision moment.

I then deliberately added `refresh_proxy_leak`, a column derived directly from the target label. The score increased to 1.000 because the model was effectively given the answer.

This is target leakage, not genuine predictive performance.

I removed `refresh_proxy_leak` after the experiment and keep 0.912 as the honest score.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitation

A key limitation of my slice is uneven GA4 coverage. In March 2026, only 413,966 of 9,841,378 rows have `ga4_data_available IS TRUE`, which is about 4.21%.

Because of this, GA4-derived features would represent only a small subset of the full panel and could introduce coverage bias. For this first feature frame, I therefore rely mainly on Search Console fields that have broader availability.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] Five plain-word contract answers completed.
- [x] Exactly three verification queries executed with visible outputs.
- [x] Availability checked explicitly with `IS TRUE`.
- [x] Five features created for the March 2026 slice.
- [x] Each feature has an “available when?” explanation.
- [x] One deliberate label-derived leakage column was added.
- [x] Leaky score increased from 0.912 to 1.000.
- [x] The leaked column was removed after the experiment.
- [x] One limitation of the slice was named.
- [x] June 2026 `_sample` was not used for label development.